In [ ]:
import random
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
import tqdm
from collections import Counter
import Loading
import VariationalAutoencoder_SemiS as engine
import copy
import torch.nn.functional as F

torch.backends.cudnn.conv.fp32_precision = 'tf32' 

In [2]:
seed = 0
random.seed(seed)
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

In [3]:
DEVICE = "cuda"
DEVICE
batch = 1024

print("device")

device


In [ ]:
train_data_path = 'train_clean_cpg.pt'
valid_data_path = 'valid_clean_cpg.pt'

In [5]:
train_d = torch.load(train_data_path)
valid_d = torch.load(valid_data_path)

In [6]:
train_d.shape

torch.Size([4539725, 4, 1024])

In [7]:
train_dataset = Loading_v4.WP2TensorDatasetV4(
    data = 'training', 
    dtype = torch.float32, 
    path = train_data_path, 
    length = train_d.shape[0], 
)

valid_dataset = Loading_v4.WP2TensorDatasetV4(
    data = 'validation', 
    dtype = torch.float32, 
    path = valid_data_path, 
    length = valid_d.shape[0], 
)

In [8]:
training_data_loader = DataLoader(
    dataset=train_dataset,
    batch_size=batch,
    shuffle=True,
    num_workers=4,
    drop_last=False,
)

valid_data_loader = DataLoader(
    dataset=valid_dataset,
    batch_size=batch,
    shuffle=False,
    num_workers=4,
    drop_last=False,
)

In [9]:

model = engine.FiberSeqVAE(
    in_channels=2,
    input_length=1024,
    latent_dim=8,
    num_classes=4,
    hidden_dims=(16, 32, 64, 128),
)

model.to(device=DEVICE, dtype=torch.float32);

model_name = str(model).split("(")[0]
print(f'Loading model {model_name}...')

num_parameters = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f'Total number of trainable parameters: {num_parameters:,}')

print('Compiling model with torch.compile...')
torch._dynamo.config.cache_size_limit = 24
opt_model = torch.compile(model)
opt_model.to(device=DEVICE, dtype=torch.float32);

Loading model FiberSeqVAE...
Total number of trainable parameters: 288,742
Compiling model with torch.compile...


In [10]:
learning_rate = 1e-4
optimizer = torch.optim.AdamW(opt_model.parameters(), lr = learning_rate) 

In [13]:
def weighted_mse(pred, target, weight, eps=1e-8):
    se = (pred - target) ** 2
    return (se * weight).sum() / (weight.sum() + eps)


def vae_loss(x_recon, x, class_logits, mu, logvar, cpg_mask, labels, beta, recon_lambda=0.01, cls_lambda=1.0, center_loss=None, center_lambda=1.0):
    alpha = 10
    bg_weight = 1
    meth_lambda = 1.0
    chrom_lambda = 1.0

    meth_weight_map = bg_weight + alpha * cpg_mask

    x_recon_meth = x_recon[:, 0:1, :]
    x_recon_chrom = x_recon[:, 1:2, :]

    x_meth = x[:, 0:1, :]
    x_chrom = x[:, 1:2, :]

    recon_loss_meth = weighted_mse(x_recon_meth, x_meth, meth_weight_map)
    recon_loss_chrom = F.mse_loss(x_recon_chrom, x_chrom, reduction="mean")

    recon_loss = recon_loss_meth * meth_lambda + recon_loss_chrom * chrom_lambda

    kl_loss = -0.5 * torch.mean(
        torch.sum(1 + logvar - mu.pow(2) - logvar.exp(), dim=1)
    )

    labels = labels.view(-1)
    labeled_mask = labels > 0

    if labeled_mask.any():
        y = labels[labeled_mask].long() - 1

        cls_loss = F.cross_entropy(
            class_logits[labeled_mask].float(),
            y
        )

        pred = class_logits[labeled_mask].argmax(dim=1)
        cls_acc = (pred == y).float().mean()
    else:
        cls_loss = torch.tensor(0.0, device=x.device)
        cls_acc = torch.tensor(0.0, device=x.device)

    if center_loss is None:
        center_loss = torch.tensor(0.0, device=x.device)

    loss = recon_lambda * recon_loss + beta * kl_loss + cls_lambda * cls_loss + center_lambda * center_loss

    return (loss, recon_loss, kl_loss, recon_loss_meth, recon_loss_chrom, cls_loss, cls_acc, center_loss, labeled_mask.sum(),)


def kl_diagnostics(mu, logvar):
    kl_per_dim = -0.5 * torch.mean(
        1 + logvar - mu.pow(2) - logvar.exp(),
        dim=0
    )

    stats = {
        "kl_per_dim_mean": kl_per_dim.mean().item(),
        "kl_per_dim_max": kl_per_dim.max().item(),
        "kl_per_dim_min": kl_per_dim.min().item(),
        "active_dims": (kl_per_dim > 0.1).sum().item(),
    }

    return stats, kl_per_dim


best_val_loss = float("inf")
patience = 5
min_delta = 1e-4
counter = 0

recon_lambda = 0.05
cls_lambda = 25.0
center_lambda = 1.0

for epoch in range(100):
    num_train_batches = len(training_data_loader)
    num_val_batches = len(valid_data_loader)

    beta = 0.1

    opt_model.train()

    epoch_loss = 0.0
    epoch_recon = 0.0
    epoch_kl = 0.0
    epoch_cls = 0.0
    epoch_acc = 0.0
    epoch_labeled = 0
    epoch_center = 0.0
    epoch_center_val = 0.0

    training_bar = tqdm.tqdm(training_data_loader)

    for batch in training_bar:
        cpg, meth, chrom, state = batch

        cpg = cpg.to(DEVICE, non_blocking=True).float()
        meth = meth.to(DEVICE, non_blocking=True).float()
        chrom = chrom.to(DEVICE, non_blocking=True).float()
        state = state.to(DEVICE, non_blocking=True).float()

        labels = state

        x = torch.cat([meth, chrom], dim=1)

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast(device_type="cuda", dtype=torch.bfloat16):
            x_recon, class_logits, mu, logvar, center_loss = opt_model(x, labels)

            (loss, recon_loss, kl_loss, recon_loss_meth, recon_loss_chrom, cls_loss, cls_acc, center_loss, n_labeled,) = vae_loss(
                x_recon=x_recon, x=x, class_logits=class_logits, mu=mu, logvar=logvar, cpg_mask=cpg, labels=labels, beta=beta, recon_lambda=recon_lambda, cls_lambda=cls_lambda, center_loss=center_loss, center_lambda=center_lambda)

        loss.backward()
        optimizer.step()

    opt_model.eval()

    epoch_loss_val = 0.0
    epoch_recon_val = 0.0
    epoch_kl_val = 0.0
    epoch_cls_val = 0.0
    epoch_acc_val = 0.0
    epoch_labeled_val = 0

    mu_mean_sum = 0.0
    mu_var_sum = 0.0
    logvar_mean_sum = 0.0
    meth_val = 0.0
    chrom_val = 0.0
    kl_per_dim_sum = None

    valid_bar = tqdm.tqdm(valid_data_loader)

    with torch.no_grad():
        for batch in valid_bar:
            cpg, meth, chrom, state = batch

            cpg = cpg.to(DEVICE, non_blocking=True).float()
            meth = meth.to(DEVICE, non_blocking=True).float()
            chrom = chrom.to(DEVICE, non_blocking=True).float()
            state = state.to(DEVICE, non_blocking=True).float()

            labels = state
            x = torch.cat([meth, chrom], dim=1)

            with torch.amp.autocast(device_type="cuda", dtype=torch.bfloat16):
                x_recon, class_logits, mu, logvar, center_loss = opt_model(x, labels)

                (loss, recon_loss, kl_loss, recon_loss_meth, recon_loss_chrom, cls_loss, cls_acc, center_loss, n_labeled,) = vae_loss(
                    x_recon=x_recon, x=x, class_logits=class_logits, mu=mu, logvar=logvar, cpg_mask=cpg, labels=labels, beta=beta, recon_lambda=recon_lambda, cls_lambda=cls_lambda, center_loss=center_loss, center_lambda=center_lambda)

            _, kl_per_dim = kl_diagnostics(mu.float(), logvar.float())

            epoch_loss_val += loss.item()
            epoch_recon_val += recon_loss.item()
            epoch_kl_val += kl_loss.item()
            epoch_cls_val += cls_loss.item()
            epoch_acc_val += cls_acc.item()
            epoch_labeled_val += n_labeled.item()
            epoch_center_val += center_loss.item()

            mu_mean_sum += mu.mean().item()
            mu_var_sum += mu.var().item()
            logvar_mean_sum += logvar.mean().item()
            meth_val += recon_loss_meth.item()
            chrom_val += recon_loss_chrom.item()

            if kl_per_dim_sum is None:
                kl_per_dim_sum = kl_per_dim.detach()
            else:
                kl_per_dim_sum += kl_per_dim.detach()

    avg_val_loss = epoch_loss_val / num_val_batches
    avg_val_recon = epoch_recon_val / num_val_batches
    avg_val_kl = epoch_kl_val / num_val_batches
    avg_val_cls = epoch_cls_val / num_val_batches
    avg_val_acc = epoch_acc_val / num_val_batches
    avg_val_center = epoch_center_val / num_val_batches

    avg_mu_mean = mu_mean_sum / num_val_batches
    avg_mu_var = mu_var_sum / num_val_batches
    avg_logvar_mean = logvar_mean_sum / num_val_batches
    avg_meth = meth_val / num_val_batches
    avg_chrom = chrom_val / num_val_batches

    avg_kl_per_dim = kl_per_dim_sum / num_val_batches

    val_stats = {
        "kl_per_dim_mean": avg_kl_per_dim.mean().item(),
        "kl_per_dim_max": avg_kl_per_dim.max().item(),
        "kl_per_dim_min": avg_kl_per_dim.min().item(),
        "active_dims": (avg_kl_per_dim > 0.1).sum().item(),
    }


    if avg_val_loss < best_val_loss - min_delta:
        best_val_loss = avg_val_loss
        counter = 0
        torch.save(model.state_dict(), "models/VAE_SS_v2.pt")
    else:
        counter += 1

    if counter >= patience:
        print(f"Early stopping at epoch {epoch + 1}")
        break

    print(
        f"Epoch {epoch+1:02d} | "
        f"beta={beta:.2f} | cls_lambda={cls_lambda:.2f} | "
        f"val_loss={avg_val_loss:.2f} | val_recon={avg_val_recon:.1f} | val_kl={avg_val_kl:.1f} | "
        f"val_cls={avg_val_cls:.2f} | val_acc={avg_val_acc:.2f} | "
        f"loss_meth={avg_meth:.2f} | loss_chrom={avg_chrom:.2f} | "
        f"KL dims: active={val_stats['active_dims']}/{opt_model.latent_dim} | "
        f"mean={val_stats['kl_per_dim_mean']:.2f} | "
        f"max={val_stats['kl_per_dim_max']:.2f} | "
        f"min={val_stats['kl_per_dim_min']:.2f} | "
        f"val_center={avg_val_center:.2f} | "
        f"mu_mean={avg_mu_mean:.2f} | "
        f"mu_var={avg_mu_var:.2f} | "
        f"logvar_mean={avg_logvar_mean:.2f}"
    )

  0%|          | 0/4434 [00:00<?, ?it/s]

100%|██████████| 355/355 [00:06<00:00, 58.24it/s]


Epoch 01 | beta=0.10 | cls_lambda=25.00 | val_loss=118.49 | val_recon=1872.6 | val_kl=6.0 | val_cls=0.90 | val_acc=0.60 | loss_meth=453.91 | loss_chrom=1418.69 | KL dims: active=8/8 | mean=0.75 | max=2.36 | min=0.39 | val_center=1.73 | mu_mean=-0.10 | mu_var=0.23 | logvar_mean=-1.24


100%|██████████| 355/355 [00:04<00:00, 81.06it/s]


Epoch 02 | beta=0.10 | cls_lambda=25.00 | val_loss=82.34 | val_recon=1178.4 | val_kl=7.5 | val_cls=0.88 | val_acc=0.61 | loss_meth=450.96 | loss_chrom=727.42 | KL dims: active=8/8 | mean=0.93 | max=2.70 | min=0.52 | val_center=0.71 | mu_mean=-0.16 | mu_var=0.16 | logvar_mean=-1.60


100%|██████████| 355/355 [00:04<00:00, 79.91it/s]


Epoch 03 | beta=0.10 | cls_lambda=25.00 | val_loss=59.69 | val_recon=722.7 | val_kl=8.1 | val_cls=0.89 | val_acc=0.61 | loss_meth=450.66 | loss_chrom=272.01 | KL dims: active=8/8 | mean=1.01 | max=2.78 | min=0.65 | val_center=0.46 | mu_mean=-0.17 | mu_var=0.13 | logvar_mean=-1.86


100%|██████████| 355/355 [01:11<00:00,  4.99it/s]


Epoch 04 | beta=0.10 | cls_lambda=25.00 | val_loss=56.69 | val_recon=666.8 | val_kl=9.4 | val_cls=0.88 | val_acc=0.60 | loss_meth=438.60 | loss_chrom=228.18 | KL dims: active=8/8 | mean=1.17 | max=2.89 | min=0.70 | val_center=0.43 | mu_mean=-0.13 | mu_var=0.11 | logvar_mean=-2.33


 89%|████████▉ | 3944/4434 [17:24<01:56,  4.22it/s]